# Haiku MIL Survival (nested cross-validation)

**Project Name:** Haiku

## Purpose
- Reproduces the colorectal cancer (CRC) survival benchmark of manuscript **Figure 3d–f** (Methods, "MIL survival analysis with Cox proportional hazards").
- Five frozen spatial-proteomics encoders are compared under one protocol: **Haiku(mIF)** (after the projection head), **KRONOS**, **Eva**, **VirTues (official)** and **VirTues (ours)**. Only the attention-based MIL Cox head is trained.
- Cohort: CRC, `FOLLOW UP (months)` with `survival_status` (66 patients, 35 patient-level events; 123 acquisitions, 64 acquisition-level events).
- Protocol:
  - 5 repeats × **3-fold** `StratifiedGroupKFold` grouped by **patient** and stratified on the event indicator (15 outer folds; each held-out fold has 22 patients and at least 11 events).
  - Hyperparameters are selected **separately for each encoder** by an inner 3-fold patient-grouped CV over an 8-point grid (embedding dim {16, 64} × learning rate {2.5e-4, 1e-3} × instance ℓ2 normalization on/off), scored by the concordance index.
  - The selected configuration is refit on the outer training split with early stopping on a further 15% patient-grouped validation split. The loss is the normalized negative Cox partial log-likelihood.
- Evaluation:
  - Each acquisition's out-of-fold risk is **averaged** over the five repeats, and one Harrell C-index is computed over all 123 acquisitions.
  - 95% CIs are patient-clustered bootstrap percentile intervals and P values are two-sided paired patient-clustered bootstrap tests against Haiku (B = 2,000).
  - Kaplan–Meier curves split patients at the median pooled out-of-fold risk (patient-level mean risk), with a log-rank test.

## Notes
- `patient_id` is derived from `SAMPLE_LABEL` by stripping a trailing `_<digit>` replicate suffix.
- Every encoder sees the same held-out acquisitions, patches and marker channels (the 81 markers shared with the KRONOS and Eva vocabularies; Methods, "Spatial-proteomics encoder baselines"). Patch embeddings are precomputed once per encoder.
- The C-index, Cox loss, Kaplan–Meier estimator and log-rank test are implemented below, so `lifelines` is not required.
- Set `RUN_TRAINING = True` to rerun the nested CV (about 1,900 small MIL heads, roughly an hour or two on one GPU). With `RUN_TRAINING = False` the notebook reads the saved out-of-fold risks from `OUTPUT_DIR` and recomputes all statistics and plots.
- Paths and checkpoints may be environment-specific. Run cells top-to-bottom and set your config paths first.
- Supersedes the earlier single-run 5-fold version of this notebook (one Haiku-tuned configuration shared by Haiku and VirTues, mean over folds, a single representative Kaplan–Meier fold).

In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
SAMPLES_JSON   = HAIKU_ROOT / 'overlap_samples_final.json'
CRC_CSV        = HAIKU_ROOT / 'downstream' / 'CRC_terminal_survival.csv'
OUTPUT_DIR     = HAIKU_ROOT / 'downstream' / 'mil_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Patch embeddings of the held-out acquisitions, one file per encoder (rows aligned
# with REGION_LABEL_FILE, which maps each patch to its acquisition).
REGION_LABEL_FILE = 'virtues_holdout_region_label.pt'
ARM_FILES = {
    'haiku_postproj':   'haikuproj_holdout_embedding.pt',
    'kronos':           'kronos_holdout_embedding.pt',
    'eva':              'eva_holdout_embedding.pt',
    'virtues_official': 'virtues_official_holdout/virtual_codex_embedding.pt',
    'virtues_ours':     'virtues_holdout_embedding.pt',
}

RUN_TRAINING = False   # True: rerun the nested CV; False: reuse saved out-of-fold risks

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)

In [ ]:
import os, re, json, copy, random, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, GroupShuffleSplit
from scipy.stats import chi2
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARMS = list(ARM_FILES)
REF = 'haiku_postproj'
LABEL = {'haiku_postproj': 'Haiku (mIF)', 'kronos': 'KRONOS', 'eva': 'Eva',
         'virtues_official': 'VirTues (official)', 'virtues_ours': 'VirTues (ours)'}


def load_arm_bags():
    # Group each encoder's patch embeddings into one bag per acquisition.
    ref_ids = sorted(json.load(open(SAMPLES_JSON)).keys())
    rl = torch.load(EMBEDDINGS_DIR / REGION_LABEL_FILE, weights_only=True).reshape(-1).numpy()
    arms = {}
    for name, fname in ARM_FILES.items():
        e = torch.load(EMBEDDINGS_DIR / fname, weights_only=True).float().numpy()
        assert len(e) == len(rl), (name, e.shape, rl.shape)
        bags = {}
        for i in range(len(rl)):
            bags.setdefault(ref_ids[int(rl[i])], []).append(e[i])
        arms[name] = {k: np.stack(v) for k, v in bags.items()}
        print(f'{LABEL[name]:20s} dim={e.shape[1]:5d}  bags={len(bags)}')
    return arms


if RUN_TRAINING:
    arm_bags = load_arm_bags()

In [ ]:
# ---- cohort ----
crc = pd.read_csv(CRC_CSV)
crc['time'] = pd.to_numeric(crc['FOLLOW UP (months)'], errors='coerce')
crc = crc[np.isfinite(crc['time'])].copy()
crc['event'] = crc['survival_status'].astype(str).str.strip().str.lower().isin(['dead', 'deceased']).astype(int)
crc['pid'] = crc['SAMPLE_LABEL'].astype(str).map(lambda s: re.sub(r'_\d+$', '', s))
A2P = dict(zip(crc.ACQUISITION_ID, crc.pid))
TT, EE = dict(zip(crc.ACQUISITION_ID, crc.time)), dict(zip(crc.ACQUISITION_ID, crc.event))
pat = crc.groupby('pid')['event'].first()
print(f'acquisitions={len(crc)} (events {crc.event.sum()})  patients={len(pat)} (events {pat.sum()})')

In [ ]:
# ---- survival metrics ----
def concordance(time, event, risk):
    # Harrell's C: a higher risk must mean a shorter survival time.
    t, e, r = np.asarray(time, float), np.asarray(event, int), np.asarray(risk, float)
    num = den = 0.0
    for i in range(len(t)):
        if e[i] != 1:
            continue
        cmp_ = t > t[i]
        den += cmp_.sum()
        num += (r[i] > r[cmp_]).sum() + 0.5 * (r[i] == r[cmp_]).sum()
    return float(num / den) if den > 0 else np.nan


def cox_partial_ll(risk, time, event):
    # Normalized negative Cox partial log-likelihood (Breslow risk sets).
    order = torch.argsort(time, descending=True)
    risk, event = risk[order], event[order]
    ll = event * (risk - torch.logcumsumexp(risk, dim=0))
    return -ll.sum() / event.sum().clamp(min=1)


def km(time, event):
    t, e = np.asarray(time, float), np.asarray(event, int)
    o = np.argsort(t, kind='mergesort'); t, e = t[o], e[o]
    ts, surv, s = [0.0], [1.0], 1.0
    for u in np.unique(t):
        at, d = (t >= u).sum(), ((t == u) & (e == 1)).sum()
        if d == 0:
            continue
        s *= 1.0 - d / at
        ts.append(float(u)); surv.append(s)
    return np.asarray(ts), np.asarray(surv)


def logrank(t1, e1, t2, e2):
    t, e = np.concatenate([t1, t2]), np.concatenate([e1, e2])
    g = np.concatenate([np.zeros(len(t1), int), np.ones(len(t2), int)])
    O1 = E1 = V = 0.0
    for u in np.unique(t[e == 1]):
        n, n1 = (t >= u).sum(), ((t >= u) & (g == 0)).sum()
        d, d1 = ((t == u) & (e == 1)).sum(), ((t == u) & (e == 1) & (g == 0)).sum()
        if n <= 1:
            continue
        O1 += d1; E1 += d * n1 / n
        V += d * (n1 / n) * (1 - n1 / n) * (n - d) / (n - 1)
    if V <= 0:
        return 0.0, 1.0
    x = (O1 - E1) ** 2 / V
    return float(x), float(chi2.sf(x, 1))

In [ ]:
# ---- MIL dataset, model, training (identical for every encoder) ----
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def grouped_train_val_split(ids, groups, stratify=None, test_size=0.1, seed=SEED):
    # Patient-aware train/val split: no patient appears in both parts.
    ids_arr, groups = np.asarray(ids), np.asarray(groups)
    if len(set(groups)) < 2:
        return ids, ids
    if stratify is not None and len(np.unique(stratify)) >= 2:
        n_splits = max(2, int(round(1.0 / test_size)))
        for nsp in (n_splits, 5, 3, 2):
            try:
                sgkf = StratifiedGroupKFold(n_splits=nsp, shuffle=True, random_state=seed)
                tr, va = next(sgkf.split(ids_arr, stratify, groups=groups))
                return ids_arr[tr].tolist(), ids_arr[va].tolist()
            except ValueError:
                continue
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr, va = next(gss.split(ids_arr, groups=groups))
    return ids_arr[tr].tolist(), ids_arr[va].tolist()


class SurvBags(Dataset):
    def __init__(self, emb, ids, l2=True):
        self.ids, self.bags, self.t, self.e = [], [], [], []
        for b in ids:
            if b not in emb or b not in TT:
                continue
            X = np.asarray(emb[b], dtype=np.float32)
            if l2:
                X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-8, None)
            self.ids.append(b); self.bags.append(torch.from_numpy(X))
            self.t.append(float(TT[b])); self.e.append(int(EE[b]))
        if not self.bags:
            raise ValueError('empty dataset')
        self.Din = self.bags[0].shape[1]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.bags[i], self.t[i], self.e[i], self.ids[i]


def collate(batch):
    bags, t, e, ids = zip(*batch)
    B, Nmax, D = len(bags), max(b.shape[0] for b in bags), bags[0].shape[1]
    X = torch.zeros(B, Nmax, D); Mk = torch.zeros(B, Nmax, dtype=torch.bool)
    for i, b in enumerate(bags):
        X[i, :b.shape[0]] = b; Mk[i, :b.shape[0]] = True
    return X, Mk, torch.tensor(t, dtype=torch.float32), torch.tensor(e, dtype=torch.float32), list(ids)


class AttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H))).squeeze(-1)
        A = torch.softmax(A.masked_fill(~mask, float('-inf')), dim=1)
        return torch.einsum('bn,bnd->bd', A, H), A


class SurvMIL(nn.Module):
    def __init__(self, d, embed_dim=64, dropout=0.1):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d, embed_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.LayerNorm(embed_dim))
        self.pool = AttnPool(embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(embed_dim, 1))

    def forward(self, X, mask):
        Z, A = self.pool(self.enc(X), mask)
        return self.head(Z).squeeze(-1), A


@torch.no_grad()
def predict(mdl, emb, ids, hp):
    dl = DataLoader(SurvBags(emb, ids, hp['l2_norm']), batch_size=256, shuffle=False, collate_fn=collate)
    mdl.eval()
    R, T, E, I = [], [], [], []
    for X, Mk, t, e, bid in dl:
        r, _ = mdl(X.to(DEVICE), Mk.to(DEVICE))
        R.append(r.cpu().numpy()); T.append(t.numpy()); E.append(e.numpy()); I += bid
    R, T, E = np.concatenate(R), np.concatenate(T), np.concatenate(E)
    return concordance(T, E, R), pd.DataFrame({'id': I, 'time': T, 'event': E, 'risk': R})


def train_surv(emb, tr_ids, va_ids, hp):
    # AdamW + cosine LR, early stopping on the validation C-index.
    set_seed(SEED)
    dtr = SurvBags(emb, tr_ids, hp['l2_norm'])
    ltr = DataLoader(dtr, batch_size=hp['batch_size'], shuffle=True, collate_fn=collate)
    mdl = SurvMIL(dtr.Din, hp['embed_dim'], hp['dropout']).to(DEVICE)
    opt = torch.optim.AdamW(mdl.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    sch = CosineAnnealingLR(opt, T_max=hp['max_epochs'], eta_min=hp['lr'] * 0.01)
    best, best_state, wait = -np.inf, None, 0
    for _ in range(hp['max_epochs']):
        mdl.train()
        for X, Mk, T, E, _ in ltr:
            if E.sum() == 0:
                continue
            r, _ = mdl(X.to(DEVICE), Mk.to(DEVICE))
            loss = cox_partial_ll(r, T.to(DEVICE), E.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        c = predict(mdl, emb, va_ids, hp)[0]
        if np.isfinite(c) and c > best:
            best, best_state, wait = c, copy.deepcopy(mdl.state_dict()), 0
        else:
            wait += 1
            if wait >= hp['patience']:
                break
    if best_state is not None:
        mdl.load_state_dict(best_state)
    return mdl

In [ ]:
# ---- nested cross-validation ----
GRID = [dict(dropout=0.1, wd=0.0, batch_size=32, max_epochs=50, patience=10,
             embed_dim=ed, lr=lr, l2_norm=ln)
        for ed, lr, ln in itertools.product([16, 64], [2.5e-4, 1e-3], [True, False])]
N_REPEATS, N_OUTER, N_INNER = 5, 3, 3


def grouped_splits(ids, ev, groups, k):
    try:
        return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED)
                    .split(ids, ev, groups=groups))
    except ValueError:
        return list(GroupKFold(n_splits=k).split(ids, groups=groups))


def inner_select(emb, ids, groups):
    # Pick one configuration for ONE encoder using only the outer training split.
    ids, g = np.asarray(ids), np.asarray(groups)
    ev = np.array([EE[i] for i in ids])
    n_eff = min(N_INNER, len(np.unique(g)), max(int(ev.sum()), 1))
    if n_eff < 2:
        return GRID[0]
    sp = grouped_splits(ids, ev, g, n_eff)
    best, best_s = GRID[0], -np.inf
    for hp in GRID:
        cs = []
        for itr, iva in sp:
            try:
                m = train_surv(emb, ids[itr].tolist(), ids[iva].tolist(), hp)
                cs.append(predict(m, emb, ids[iva].tolist(), hp)[0])
            except Exception:
                cs.append(np.nan)
        s = np.nanmean(cs) if not np.all(np.isnan(cs)) else -np.inf
        if s > best_s:
            best_s, best = s, hp
    return best


def run_nested(arm_bags):
    ids = sorted(set(TT) & set(A2P) & set.intersection(*[set(b) for b in arm_bags.values()]))
    groups = np.array([A2P[i] for i in ids]); ev = np.array([EE[i] for i in ids])
    rows = []
    for rep in range(N_REPEATS):
        try:
            outer = list(StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=1000 + rep)
                         .split(ids, ev, groups=groups))
        except ValueError:
            outer = list(GroupKFold(n_splits=N_OUTER).split(ids, groups=groups))
        for fold, (tr, te) in enumerate(outer):
            assert not set(groups[tr]) & set(groups[te]), 'patient leakage'
            tr_ids, te_ids = [ids[i] for i in tr], [ids[i] for i in te]
            for arm in ARMS:
                emb = arm_bags[arm]
                hp = inner_select(emb, tr_ids, groups[tr])
                sub_tr, sub_va = grouped_train_val_split(tr_ids, groups[tr], stratify=ev[tr],
                                                         test_size=0.15, seed=SEED)
                mdl = train_surv(emb, sub_tr, sub_va, hp)
                _, pred = predict(mdl, emb, te_ids, hp)
                pred['arm'], pred['rep'], pred['fold'] = arm, rep, fold
                pred['embed_dim'], pred['lr'], pred['l2_norm'] = hp['embed_dim'], hp['lr'], hp['l2_norm']
                rows.append(pred)
            print(f'  repeat {rep} fold {fold} done')
    return pd.concat(rows, ignore_index=True)


oof_path = OUTPUT_DIR / 'mil_surv_crc_oof_risk.csv'
if RUN_TRAINING:
    run_nested(arm_bags).to_csv(oof_path, index=False)
OOF = pd.read_csv(oof_path)
OOF['pid'] = OOF['id'].map(A2P)
print(f'{len(OOF[OOF.arm == REF])} out-of-fold risks per encoder '
      f'({OOF.rep.nunique()} repeats x {OOF.groupby(["rep", "fold"]).ngroups // OOF.rep.nunique()} folds)')

In [ ]:
# ---- pooled C-index with patient-clustered bootstrap ----
N_BOOT, BOOT_SEED = 2000, 0

# average each acquisition's risk over the repeats
POOLED = {a: OOF[OOF.arm == a].groupby('id').agg(time=('time', 'first'), event=('event', 'first'),
                                                  risk=('risk', 'mean'), pid=('pid', 'first')).reset_index()
          for a in ARMS}


def boot_indices(pid, rng):
    uq = np.unique(pid); by = {g: np.where(pid == g)[0] for g in uq}
    return np.concatenate([by[g] for g in rng.choice(uq, len(uq), replace=True)])


rows = []
for a in ARMS:
    p = POOLED[a]
    rng = np.random.default_rng(BOOT_SEED)
    vals = [concordance(p.time.values[i], p.event.values[i], p.risk.values[i])
            for i in (boot_indices(p.pid.values, rng) for _ in range(N_BOOT))]
    vals = [v for v in vals if np.isfinite(v)]
    row = dict(encoder=LABEL[a], c_index=concordance(p.time, p.event, p.risk),
               ci_lo=np.percentile(vals, 2.5), ci_hi=np.percentile(vals, 97.5),
               per_repeat_mean=np.nanmean([concordance(g.time, g.event, g.risk)
                                           for _, g in OOF[OOF.arm == a].groupby('rep')]))
    if a != REF:
        m = POOLED[REF].merge(p, on='id', suffixes=('_o', '_v'))
        rng = np.random.default_rng(BOOT_SEED)
        ds = []
        for _ in range(N_BOOT):
            i = boot_indices(m.pid_o.values, rng)
            c1 = concordance(m.time_o.values[i], m.event_o.values[i], m.risk_o.values[i])
            c2 = concordance(m.time_o.values[i], m.event_o.values[i], m.risk_v.values[i])
            if np.isfinite(c1) and np.isfinite(c2):
                ds.append(c1 - c2)
        ds = np.asarray(ds)
        row.update(delta_haiku_minus=concordance(m.time_o, m.event_o, m.risk_o)
                   - concordance(m.time_o, m.event_o, m.risk_v),
                   delta_lo=np.percentile(ds, 2.5), delta_hi=np.percentile(ds, 97.5),
                   p_vs_haiku=min(1.0, 2 * min((ds <= 0).mean(), (ds >= 0).mean())))
    rows.append(row)

cindex = pd.DataFrame(rows)
cindex.to_csv(OUTPUT_DIR / 'mil_surv_crc_cindex.csv', index=False)
pd.options.display.float_format = '{:.3f}'.format
cindex

In [ ]:
# ---- figures: C-index bars and Kaplan-Meier curves at the median pooled risk ----
COLORS = {'haiku_postproj': '#2F5A8A', 'kronos': '#7FA7C9', 'eva': '#9BB89A',
          'virtues_official': '#C9A27F', 'virtues_ours': '#B5736B'}

fig, ax = plt.subplots(figsize=(4.5, 3.6))
x = np.arange(len(ARMS))
ax.bar(x, cindex.c_index, color=[COLORS[a] for a in ARMS], edgecolor='black', linewidth=0.5,
       yerr=[cindex.c_index - cindex.ci_lo, cindex.ci_hi - cindex.c_index], capsize=3)
ax.axhline(0.5, color='k', ls='--', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([LABEL[a] for a in ARMS], rotation=30, ha='right')
ax.set_ylim(0.3, 0.9); ax.set_ylabel('Pooled out-of-fold C-index')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
for ext in ('svg', 'pdf', 'png'):
    fig.savefig(OUTPUT_DIR / f'mil_surv_crc_cindex.{ext}', dpi=300, bbox_inches='tight')
plt.show()

km_rows = []
fig, axes = plt.subplots(1, len(ARMS), figsize=(3.4 * len(ARMS), 3.2), sharey=True)
for ax, a in zip(axes, ARMS):
    # patient-level risk = mean of that patient's pooled acquisition risks
    pt = POOLED[a].groupby('pid').agg(time=('time', 'first'), event=('event', 'first'),
                                      risk=('risk', 'mean')).reset_index()
    med = np.median(pt.risk)
    lo, hi = pt[pt.risk <= med], pt[pt.risk > med]
    _, p = logrank(lo.time.values, lo.event.values, hi.time.values, hi.event.values)
    for g, name, ls in [(lo, 'Low risk', '-'), (hi, 'High risk', '--')]:
        t, s = km(g.time.values, g.event.values)
        ax.step(t, s, where='post', color=COLORS[a], ls=ls,
                label=f'{name} (n={len(g)}, events={int(g.event.sum())})')
    ax.set_title(f'{LABEL[a]}\nlog-rank P = {p:.2g}', fontsize=10)
    ax.set_xlabel('Months'); ax.legend(frameon=False, fontsize=7, loc='lower left')
    ax.spines[['top', 'right']].set_visible(False)
    km_rows.append(dict(encoder=LABEL[a], n_low=len(lo), events_low=int(lo.event.sum()),
                        n_high=len(hi), events_high=int(hi.event.sum()), logrank_p=p))
axes[0].set_ylabel('Survival probability')
plt.tight_layout()
for ext in ('svg', 'pdf', 'png'):
    fig.savefig(OUTPUT_DIR / f'mil_surv_crc_km.{ext}', dpi=300, bbox_inches='tight')
plt.show()
pd.DataFrame(km_rows)